# 20/ Product Recommendation System

**Dataset (Kaggle):** [https://www.kaggle.com/datasets/carrie1/ecommerce-data](https://www.kaggle.com/datasets/carrie1/ecommerce-data)

> UK online retailer transactions: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.

**Goal:** Recommend products to a customer based on co-purchase patterns (market basket analysis) and past purchase history.

**How to get the data:**
1. Download from the link above (or run `kaggle datasets download -d <dataset-slug> -p data/ --unzip`, requires a `kaggle.json` API token in `~/.kaggle/`).
2. Place the extracted file(s) in a local `data/` folder next to this notebook.
3. Update `DATA_PATH` in the cell below if your filename differs.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")


In [ ]:
df = pd.read_csv("data/data.csv", encoding='latin-1')
df = df.dropna(subset=['Description', 'CustomerID'])
print(df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
print("Customers:", df['CustomerID'].nunique(), " Products:", df['StockCode'].nunique())
top_products = df['Description'].value_counts().head(15)
top_products.plot(kind='barh', figsize=(8,6))
plt.gca().invert_yaxis()
plt.title("Top 15 most purchased products")
plt.show()


## Build Customer-Product Matrix

In [ ]:
basket = df.groupby(['CustomerID', 'StockCode'])['Quantity'].sum().unstack(fill_value=0)
basket_binary = (basket > 0).astype(int)
print(basket_binary.shape)


## Item-Based Similarity (Co-purchase patterns)

In [ ]:
item_similarity = cosine_similarity(basket_binary.T)
product_map = df.drop_duplicates('StockCode').set_index('StockCode')['Description']
item_sim_df = pd.DataFrame(item_similarity, index=basket_binary.columns, columns=basket_binary.columns)

def recommend_products(stock_code, top_n=10):
    if stock_code not in item_sim_df.columns:
        return "Product not found."
    similar = item_sim_df[stock_code].sort_values(ascending=False)[1:top_n+1]
    return pd.DataFrame({'StockCode': similar.index, 'Description': [product_map.get(c, '?') for c in similar.index], 'Similarity': similar.values})

sample_code = basket_binary.columns[0]
print("Because you bought:", product_map.get(sample_code, '?'))
recommend_products(sample_code)


## Recommend for a Specific Customer

In [ ]:
def recommend_for_customer(customer_id, top_n=10):
    if customer_id not in basket_binary.index:
        return "Customer not found."
    purchased = basket_binary.loc[customer_id]
    purchased_items = purchased[purchased > 0].index
    scores = item_sim_df[purchased_items].sum(axis=1)
    scores = scores.drop(purchased_items, errors='ignore')
    top = scores.sort_values(ascending=False).head(top_n)
    return pd.DataFrame({'StockCode': top.index, 'Description': [product_map.get(c, '?') for c in top.index], 'Score': top.values})

sample_customer = basket_binary.index[0]
recommend_for_customer(sample_customer)


## Next Steps
- Run Apriori/FP-Growth (`mlxtend.frequent_patterns`) for classic market-basket association rules.
- Weight by quantity/revenue instead of binary purchased/not.
- Add a popularity-based fallback for new customers (cold start).